# Signal Denoising with the Q.ANT NPU

## Overview

This notebook shows how to denoise a noisy sine wave in the frequency domain:

1. Generate a sine wave and add random noise
2. Compute its **discrete Fourier transform (DFT) on the Q.ANT NPU** with `q_native.dft`
3. Apply a very simple low-pass filter by setting all frequency bins above a cutoff to zero
4. Transform back to the time domain with the **inverse DFT, also computed on the Q.ANT NPU**

### Prerequisites
Install the requirements listed in the requirements.txt file in the folder.
```bash
pip install -r requirements.txt
```

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import qant_native_computing_toolkit.native as q_native
from ml_dtypes import bcomplex32

# fix random seed for reproducibility
rng = np.random.default_rng(0)

## 1. Create a noisy sine wave

We sample a 5 Hz sine wave for one second at 1024 samples per second and add Gaussian noise.

In [ ]:
n_samples = 1024  # number of samples (= DFT size)
sampling_rate = 1024.0  # samples per second [Hz]
signal_freq = 5.0  # frequency of the sine wave [Hz]
noise_std = 1.5  # standard deviation of the noise

t = np.arange(n_samples) / sampling_rate
clean_signal = np.sin(2 * np.pi * signal_freq * t)
noisy_signal = clean_signal + rng.normal(0.0, noise_std, size=n_samples)

plt.figure(figsize=(10, 4))
plt.plot(t, noisy_signal, label="noisy signal", alpha=0.7)
plt.plot(t, clean_signal, label="clean sine", linewidth=2)
plt.xlabel("time [s]")
plt.ylabel("amplitude")
plt.title("Time domain: noisy input")
plt.legend()
plt.show()

## 2. DFT on the Q.ANT NPU

`q_native.dft` expects complex input of dtype `bcomplex32` (bfloat16 real and imaginary parts)
and computes the unnormalised DFT along the last axis. Our signal is real, so the imaginary part is zero.

In [ ]:
spectrum = q_native.dft(noisy_signal.astype(bcomplex32))

# convert to complex64 for further processing with numpy
spectrum = spectrum.astype(np.complex64)

# frequency of each DFT bin (in the standard FFT ordering: 0, positive, then negative frequencies)
freqs = np.fft.fftfreq(n_samples, d=1 / sampling_rate)


def plot_spectrum(spectrum, title):
    # fftshift reorders the bins from the most negative to the most positive frequency for plotting
    plt.figure(figsize=(10, 4))
    plt.plot(np.fft.fftshift(freqs), np.fft.fftshift(np.abs(spectrum)))
    plt.xlabel("frequency [Hz]")
    plt.ylabel("magnitude (unnormalised)")
    plt.title(title)
    plt.show()


plot_spectrum(spectrum, "Frequency domain: DFT of the noisy signal (Q.ANT NPU)")

The two peaks at ±5 Hz are the sine wave. The noise is spread evenly over all frequencies.

## 3. A very simple low-pass filter

We keep all frequency bins with |f| <= `cutoff_freq` and set all other bins to zero.

In [ ]:
cutoff_freq = 6.0  # [Hz]

lowpass_mask = np.abs(freqs) <= cutoff_freq
filtered_spectrum = spectrum * lowpass_mask

plot_spectrum(
    filtered_spectrum,
    f"Frequency domain: after low-pass filter (cutoff = {cutoff_freq} Hz)",
)

## 4. Back to the time domain (inverse DFT on the Q.ANT NPU)

There is no separate inverse DFT on the NPU, but we don't need one. The inverse DFT can be computed with the forward DFT using complex conjugation:

$$\text{IDFT}(X) = \frac{1}{N} \, \overline{\text{DFT}\left(\overline{X}\right)}$$

So we conjugate the filtered spectrum, run `q_native.dft` on it, conjugate the result again and divide by $N$.
The imaginary part of the result is only numerical noise (the input signal was real), so we keep the real part.

In [ ]:
def idft_on_npu(spectrum):
    """Inverse DFT computed with the forward DFT of the NPU: IDFT(X) = conj(DFT(conj(X))) / N"""
    n = spectrum.shape[-1]
    result = q_native.dft(np.conj(spectrum).astype(bcomplex32))
    return np.conj(result.astype(np.complex64)) / n


denoised_signal = idft_on_npu(filtered_spectrum).real

plt.figure(figsize=(10, 4))
plt.plot(t, noisy_signal, label="noisy signal", alpha=0.3)
plt.plot(t, clean_signal, label="clean sine", linewidth=2)
plt.plot(t, denoised_signal, label="denoised signal", linewidth=2, linestyle="--")
plt.xlabel("time [s]")
plt.ylabel("amplitude")
plt.title("Time domain: denoised result")
plt.legend()
plt.show()

mse_noisy = np.mean((noisy_signal - clean_signal) ** 2)
mse_denoised = np.mean((denoised_signal - clean_signal) ** 2)
print(f"MSE noisy vs. clean:    {mse_noisy:.4f}")
print(f"MSE denoised vs. clean: {mse_denoised:.4f}")